In [ ]:
!pip install -qqq openai  rouge-score bert-score python-dotenv pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.4/362.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 91.5 kB/s eta 0:00:00


In [ ]:
import openai
import json
import pandas as pd
import openai
from dotenv import load_dotenv
import os
import torch
import bert_score
from rouge_score import rouge_scorer
from tqdm import tqdm
import multiprocessing
import shutil
from transformers import AutoModel, AutoTokenizer
import random

### *Drive Mounting*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/My Drive/Dissertation/SRoy_Dissertation/Simplification/Data

/content/drive/My Drive/Dissertation/Simplification/Data


In [ ]:
#@title Function to calculate ROUGE scores

# Initialize the ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_rouge_scores(reference, hypothesis):
    scores = scorer.score(reference, hypothesis)
    return {
        'rouge1_precision': scores['rouge1'].precision,
        'rouge1_recall': scores['rouge1'].recall,
        'rouge1_f1': scores['rouge1'].fmeasure,
        'rouge2_precision': scores['rouge2'].precision,
        'rouge2_recall': scores['rouge2'].recall,
        'rouge2_f1': scores['rouge2'].fmeasure,
        'rougeL_precision': scores['rougeL'].precision,
        'rougeL_recall': scores['rougeL'].recall,
        'rougeL_f1': scores['rougeL'].fmeasure
    }

In [ ]:
#@title Function for intiating the openai model
def get_response(system_prompt, user_prompt):
    """Get the response from the OpenAI API."""
    os.environ["OPENAI_API_KEY"] = 'sk-None-j35N8fYBqY5q2rcuySoFT3BlbkFJYhf8phIdKSsQ7aliiifd'
    client = openai.OpenAI()
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo-1106",
        messages=[{"role": "system", "content": system_prompt},
         {"role": "user", "content": user_prompt},
                  ])
    res = completion.choices[0].message.content
    return res

In [ ]:
#@title Function for generating the simplified text with prompting
def generate(legal_text, prompt, shot):
  """Generate the response from the OpenAI API."""

  # For zero, one, few shot learning
  prompt = pd.read_csv(prompt)
  raw_text = prompt[prompt.columns[0]].tolist()
  simplified_text = prompt[prompt.columns[1]].tolist()

  # Select a random index
  random_index = random.randint(0, len(raw_text) - 1)

  # Select a random pair of raw and simplified text
  selected_raw_text = raw_text[random_index]
  selected_simplified_text = simplified_text[random_index]

  # Give general instructions in the system prompt
  system_prompt = """You are a legal expert."""

  if shot == 0:
    # Give more specific instructions in the user prompt and also provide the judgement text
    user_prompt = """Please simplify the given legal text:
    '''
    {}
    '''
    """.format(legal_text)

  elif shot == 1:
    # Give more specific instructions in the user prompt and also provide the judgement text
    user_prompt = f"""Below is an example of a complex legal text that has been simplified by legal experts, so that a layman person can understand. Follow the given tips to simplify legal text:
  Raw text:
  {selected_raw_text}
  Simplified text:
  {selected_simplified_text}
  Now, please simplify the following legal text
  {legal_text}
  """

  elif shot == 2:
    # Give more specific instructions in the user prompt and also provide the judgement text
    user_prompt = f"""Below is an example of a legal text that has been simplified by legal experts. Follow the given tips to simplify legal text:
  1. Raw text: {raw_text[0]}
  Simplified text: {simplified_text[0]}

  2. Raw text: {raw_text[1]}
  Simplified text: {simplified_text[1]}

  Now, please simplify the following legal text
  {legal_text}
  """

  elif shot == 3:
    # Give more specific instructions in the user prompt and also provide the judgement text
    user_prompt = f"""Below is an example of a legal text that has been simplified by legal experts. Follow the given tips to simplify legal text:
  1. Raw text: {raw_text[0]}
  Simplified text: {simplified_text[0]}

  2. Raw text: {raw_text[1]}
  Simplified text: {simplified_text[1]}

  3. Raw text: {raw_text[2]}
  Simplified text: {simplified_text[2]}

  Now, please simplify the following legal text
  {legal_text}
  """
  else:
    raise ValueError("Invalid shot value. Please choose a value between 0 and 3.")

  response = get_response(system_prompt, user_prompt)

  # Get the response from the OpenAI API
  return response

In [ ]:
#@title Function for generating and saving multiple responses
def generate_multi_responses_and_saving(path, prompt_path, shot, output_csv):
    """Generate the response from the OpenAI API and save to CSV."""
    try:
        # Load the JSON file
        with open(path, 'r', encoding='utf-8') as f:
            text = json.load(f)

        # Prepare an empty DataFrame to collect results
        all_responses = pd.DataFrame(columns=['legal_text', 'simplified_text'])

        # Flatten the dictionary to get a list of text items
        items = [(id, value, judgement_text) for id, values in text.items() for value, judgement_text in values.items()]

        # Use tqdm to show progress
        for id, value, judgement_text in tqdm(items, desc="Processing legal texts", unit="text"):
            try:
                # Generate simplified text
                response = generate(judgement_text, prompt_path, shot)

                # Collect responses in a DataFrame
                df = pd.DataFrame({'legal_text': [judgement_text], 'simplified_text': [response]})
                all_responses = pd.concat([all_responses, df], ignore_index=True)
            except Exception as e:
                print(f"Error generating response for id {id}: {e}")

        # Save the collected DataFrame to CSV
        all_responses.to_csv(output_csv, index=False)
        print(f"\n\nData has been successfully saved to {output_csv}")

    except Exception as e:
        print(f"Error processing the file: {e}")

In [ ]:
#@title Function for converting all txt files into one single JSON file
import os
import json

def read_text_files(folder_path):
    data = {}
    file_counter = 0

    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                text = file.read()
                data[str(file_counter)] = {"text": text}
                file_counter += 1

    return data

def save_to_json(data, output_path):
    with open(output_path, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4)

## *Generating dataset for UK Legal Text*

In [ ]:
# Converting txt files into one single json file
def main():
    folder_path = 'UK_dataset'
    output_path = 'legal_text_UK.json'

    data = read_text_files(folder_path)
    save_to_json(data, output_path)

if __name__ == '__main__':
    main()

### *Generating the simplified legal texts*

In [ ]:
output_csv = "legal_text_UK_simplified_351.csv"
shot = 1
generate_multi_responses_and_saving('legal_text_UK.json', 'prompt.csv', shot, output_csv)

Processing legal texts: 100%|██████████| 351/351 [18:17<00:00,  3.13s/text]



Data has been successfully saved to legal_text_UK_simplified_351.csv


In [ ]:
uk_351 = pd.read_csv("legal_text_UK_simplified_351.csv")
uk_600 = pd.read_csv("legal_text_UK_simplified_600.csv")
uk_351.shape, uk_600.shape

((351, 2), (175, 2))

In [ ]:
# Previously generated simplified legal texts
uk_600 = pd.read_csv("legal_text_UK_simplified_600.csv")
# Newly generated simplified legal texts
uk_351 = pd.read_csv("legal_text_UK_simplified_351.csv")

# Append the new records to the existing DataFrame
uk_df = pd.concat([uk_600, uk_351], ignore_index=False)
# Saving the records
uk_df.to_csv("legal_text_UK_simplified.csv", index=False)

# Shape of the dataset
uk_df.shape

(526, 2)

## *Generating dataset for Indian Legal Texts*

In [ ]:
# Converting txt files into one single json file
def main():
    folder_path = 'IND_dataset'
    output_path = 'legal_text_IND.json'

    data = read_text_files(folder_path)
    save_to_json(data, output_path)

if __name__ == '__main__':
    main()

### *Generating the simplified legal texts*

In [ ]:
output_csv = "legal_text_IND_simplified.csv"
shot = 1
generate_multi_responses_and_saving('legal_text_IND.json', 'prompt.csv', shot, output_csv)

Processing legal texts: 100%|██████████| 807/807 [21:38<00:00,  1.61s/text]



Data has been successfully saved to legal_text_IND_simplified.csv


In [ ]:
df = pd.read_csv("legal_text_IND_simplified.csv")
df.head()

,legal_text,simplified_text
0,1. (1) This Act may be called the Bharatiya Na...,Title of the Act: The Bharatiya Nagarik Suraks...
1,"3. (1) Unless the context otherwise requires, ...",Simplified text:\nReferring to Magistrates: In...
2,4. (1) All offences under the Bharatiya Nyaya ...,Processing Offences: All crimes under the Bhar...
3,"5. Nothing contained in this Sanhita shall, in...",Nothing in this collection of laws will change...
4,6. BesidestheHighCourtsandtheCourtsconstituted...,In addition to the High Courts and other court...


## *Generating dataset for USA Legal Texts*

In [ ]:
# Converting txt files into one single json file
def main():
    folder_path = 'USA_dataset'
    output_path = 'legal_text_USA.json'

    data = read_text_files(folder_path)
    save_to_json(data, output_path)

if __name__ == '__main__':
    main()

### *Generating the simplified legal texts*

In [ ]:
output_csv = "legal_text_USA_simplified.csv"
shot = 1
generate_multi_responses_and_saving('legal_text_USA.json', 'prompt.csv', shot, output_csv)

Processing legal texts: 100%|██████████| 794/794 [22:53<00:00,  1.73s/text]



Data has been successfully saved to legal_text_USA_simplified.csv


In [ ]:
df = pd.read_csv("legal_text_USA_simplified.csv")
df.shape

(793, 2)